## Concept 5 — RAG Agent

### What is it?
A RAG Agent wraps a vectorstore retriever as a **tool**. The agent decides *when* to search —
only retrieving documents when the query actually needs them. This is smarter than a fixed
RAG pipeline that always retrieves regardless of the question.

### Pipeline
```
Query: 'What is 144/12?'        → Agent: 'Math only → call calculate'
Query: 'What is an embedding?'  → Agent: 'Need docs → call search_docs_rag (vectorstore)'
Query: 'Search docs + calc'     → Agent: 'Both → call both tools'
```

### RAG Pipeline vs RAG Agent
```
RAG Pipeline:  ALWAYS retrieves → even '2+2?' goes through vectorstore (wasteful)
RAG Agent:     decides WHEN to retrieve → math queries skip vectorstore entirely
```

### Limitation (what Concept 6 fixes)
One agent handles everything — no specialization per task type.
→ Concept 6 routes queries to specialized sub-agents.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, calculate, get_weather, TEST_QUERIES, run_queries
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

llm = get_llm()
print('LLM ready — setting up RAG vectorstore...')

### Step 1 — Build the RAG Vectorstore
Load real LangSmith documentation, split into chunks, embed into FAISS.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print('Loading LangSmith docs...')
loader = WebBaseLoader('https://docs.langchain.com/langsmith/agent-server')
docs   = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks   = splitter.split_documents(docs)

embeddings  = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 3})

print(f'Vectorstore ready: {len(docs)} page(s) → {len(chunks)} chunks')

### Step 2 — Wrap Retriever as a Tool
The retriever becomes a first-class tool the agent can call like any other function.

In [ ]:
@tool
def search_docs_rag(query: str) -> str:
    """Search the LangSmith documentation for detailed information about agents, deployment, persistence, and LangChain concepts."""
    docs = retriever.invoke(query)
    if not docs:
        return 'No relevant documentation found.'
    return '\n\n'.join(doc.page_content[:400] for doc in docs)

# All tools: real RAG + math + weather
rag_tools = [search_docs_rag, calculate, get_weather]
print(f'RAG tools registered: {[t.name for t in rag_tools]}')

### Step 3 — Build RAG Agent
Bind all tools. The agent dynamically picks search vs calculate vs weather per query.

In [ ]:
llm_with_rag = llm.bind_tools(rag_tools)
tool_map     = {t.name: t for t in rag_tools}

def run_rag_agent(query: str) -> str:
    messages = [HumanMessage(content=query)]
    response = llm_with_rag.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            print(f'  [Tool] {call["name"]} → {str(result)[:120]}...')
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        final = llm_with_rag.invoke(messages)
        return final.content

    return response.content

### Step 4 — Compare: When Does the Agent Search?
Watch which tool gets called for different query types.

In [ ]:
test_cases = [
    ('Math only',    'What is 144 divided by 12?'),
    ('Docs only',    'Explain LangSmith deployment'),
    ('Weather only', 'What is the weather in Bangalore?'),
    ('Mixed',        'Explain LangSmith checkpoints and calculate 8 times 7'),
]

for label, q in test_cases:
    print(f'\n[{label}] Q: {q}')
    answer = run_rag_agent(q)
    print(f'A: {answer[:200]}...' if len(answer) > 200 else f'A: {answer}')
    print('-' * 40)

### Full Function

In [ ]:
def rag_agent(query: str) -> str:
    return run_rag_agent(query)

### Test — Standard Queries

In [ ]:
run_queries(rag_agent, label='Concept 5 — RAG Agent')